# 03 — Sentence-Transformer Embeddings

Encode every resume chunk (from notebook 02) into a dense 384-dim vector using `all-MiniLM-L6-v2`.

**Input:** `data/processed/resume_chunks.json` — flat list of chunks with `chunk_id`, `candidate_id`, `section_type`, `text`  
**Outputs:**
- `data/processed/chunk_embeddings.npy` — (N_chunks, 384) float32 array, rows aligned to chunks file
- `data/processed/chunks_with_embeddings.json` — chunk metadata aligned by index with the .npy rows

These two files feed into **04_1** (Qdrant) and **04_2** (FAISS) for indexing and search.

In [ ]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch

PROJECT_ROOT  = Path("../").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CHUNKS_INPUT  = PROCESSED_DIR / "resume_chunks.json"
EMB_PATH      = PROCESSED_DIR / "chunk_embeddings.npy"
META_PATH     = PROCESSED_DIR / "chunks_with_embeddings.json"

MODEL_NAME  = "all-MiniLM-L6-v2"
BATCH_SIZE  = 64

print(f"Input  : {CHUNKS_INPUT}")
print(f"Outputs: {EMB_PATH}")
print(f"         {META_PATH}")
print(f"Model  : {MODEL_NAME}")
print(f"Device : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1 — Load chunks

In [ ]:
with open(CHUNKS_INPUT, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks from {CHUNKS_INPUT.name}")

# Section distribution
from collections import Counter
section_counts = Counter(c["section_type"] for c in chunks)
print(f"\n{'Section':<20} {'Chunks':>8}")
print("-" * 30)
for sec, cnt in sorted(section_counts.items(), key=lambda x: -x[1]):
    print(f"{sec:<20} {cnt:>8}")

# Source distribution
source_counts = Counter(c["source"] for c in chunks)
print(f"\n{'Source':<20} {'Chunks':>8}")
print("-" * 30)
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"{src:<20} {cnt:>8}")

## 2 — Load the model

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded: {MODEL_NAME}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 3 — Encode all chunks

Each chunk's `text` field is embedded independently. Because chunks are short
(one job entry, a skill list, etc.) they fit comfortably within the model's
256-token window — no truncation worries compared to whole-document embedding.

In [ ]:
texts = [c["text"] for c in chunks]

print(f"Encoding {len(texts)} chunks (batch_size={BATCH_SIZE}) ...")
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,  # L2-normalize → dot product == cosine similarity
)

embeddings = np.array(embeddings, dtype=np.float32)
print(f"\nEmbeddings shape : {embeddings.shape}")
print(f"Norm of first vec: {np.linalg.norm(embeddings[0]):.4f}  (should be ~1.0)")

## 4 — Save embeddings & aligned metadata

In [ ]:
# Save the raw embedding matrix — row i corresponds to chunks[i]
np.save(EMB_PATH, embeddings)
print(f"Saved embeddings : {EMB_PATH}  ({EMB_PATH.stat().st_size / 1024:.1f} KB)")

# Save chunk metadata (everything except the raw text, which lives in resume_chunks.json)
# The .npy row index is embedded as `emb_idx` for easy cross-referencing.
meta_records = []
for i, chunk in enumerate(chunks):
    meta_records.append({
        "emb_idx":      i,
        "chunk_id":     chunk["chunk_id"],
        "candidate_id": chunk["candidate_id"],
        "source":       chunk["source"],
        "section_type": chunk["section_type"],
        "text_preview": chunk["text"][:300],
        "metadata":     chunk.get("metadata", {}),
    })

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta_records, f, indent=2, ensure_ascii=False)

print(f"Saved chunk meta : {META_PATH}  ({META_PATH.stat().st_size / 1024:.1f} KB)")
print(f"\nTotal chunks embedded: {len(meta_records)}")

## 5 — Sanity check: nearest-neighbour search over chunks

Pick a sample chunk and find the most semantically similar chunks across all resumes.

In [ ]:
# Pick the first experience chunk as the query
query_idx = next(i for i, c in enumerate(chunks) if c["section_type"] == "experience")
query_vec = embeddings[query_idx]

sims = embeddings @ query_vec  # cosine similarities (already L2-normalized)
top_k = 6
top_indices = np.argsort(sims)[::-1][:top_k]

print(f"Query chunk ({chunks[query_idx]['candidate_id']}):")
print(f"  [{chunks[query_idx]['section_type']}] {chunks[query_idx]['text'][:120]}")
print(f"\nTop-{top_k} nearest chunks:")
print(f"{'Rank':<5} {'Score':<7} {'Section':<16} {'Candidate':<40} {'Preview'}")
print("-" * 100)
for rank, idx in enumerate(top_indices):
    c = chunks[idx]
    print(f"{rank:<5} {sims[idx]:.4f}  {c['section_type']:<16} {c['candidate_id']:<40} {c['text'][:60]}")

## 6 — (Optional) Visualize chunk embedding clusters

Reduce to 2D with t-SNE and colour by section type to verify that sections cluster as expected.

In [ ]:
# TODO: Uncomment and run once you have enough chunks to make this interesting.
#
# from sklearn.manifold import TSNE
# import matplotlib.pyplot as plt
#
# # Sample for speed — t-SNE is O(n²)
# sample_n = min(2000, len(embeddings))
# indices = np.random.choice(len(embeddings), sample_n, replace=False)
# coords = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(embeddings[indices])
#
# section_types = [chunks[i]["section_type"] for i in indices]
# unique_sections = sorted(set(section_types))
#
# plt.figure(figsize=(12, 8))
# for sec in unique_sections:
#     mask = [j for j, s in enumerate(section_types) if s == sec]
#     plt.scatter(coords[mask, 0], coords[mask, 1], label=sec, alpha=0.5, s=10)
# plt.legend(markerscale=2)
# plt.title(f"Chunk embeddings by section type (t-SNE, n={sample_n})")
# plt.tight_layout()
# plt.show()